1. What FastAPI actually is and what problems it solves.
2. How a client, server, HTTP, ASGI, and Uvicorn work together.
3. What FastAPI(), decorators, endpoints, and route handlers actually mean.
4. How GET, POST, PUT, PATCH, and DELETE work.
5. How path, query, and request-body parameters differ.
6. How FastAPI uses Python type hints for validation.
7. How request execution works internally.
8. How to build and test a small invoice-processing API.

In [ ]:
from fastapi import FastAPI
app=FastAPI()

@app.get('/') 
def home():
    return{
        "message": "Welcome to financial AI"
    }

In [1]:
from fastapi import FastAPI
app=FastAPI()

@app.get('/')
def home():
    return{
        "Welcome"
    }

In [ ]:
# @app.get('/') Python decorator registers a function 
# as the handler for an HTTP operation

HTTP Methods

1. GET

In [2]:
from fastapi import FastAPI
app=FastAPI()

invoices={
    101: {
        "vendor": "Aira AI",
        "amount": 12000,
        "status": "pending"
    },
    102:{
        "vendor": "Code AI",
        "amount": 13000,
        "status": "approved"

    }
}

In [ ]:
# # GET /invoices
# fast api calles the get_invoices() and 
# retrurns the dictionary as json

GET /invoices/101

In [3]:
from fastapi import HTTPException


@app.get("/invoices/{invoice_id}")
def get_invoice(invoice_id: int):
    if invoice_id not in invoices:
        raise HTTPException(
            status_code=404,
            detail="Invoice not found"
        )

    return invoices[invoice_id]

GET /invoices/filter?status=approved

In [7]:
@app.get("/invoices/filter")
def filter_invoices(status: str = "pending"):
    return {
        invoice_id: data
        for invoice_id, data in invoices.items()
        if data["status"] == status
    }

2. POST

In [8]:
from pydantic import BaseModel, Field


class InvoiceCreate(BaseModel):
    vendor: str = Field(min_length=2)
    amount: float = Field(gt=0)


@app.post("/invoices")
def create_invoice(invoice: InvoiceCreate):
    new_id = max(invoices, default=100) + 1

    invoices[new_id] = {
        "vendor": invoice.vendor,
        "amount": invoice.amount,
        "status": "pending"
    }

    return {
        "invoice_id": new_id,
        "data": invoices[new_id]
    }

In [9]:
{
  "vendor": "New Company",
  "amount": 25000
}

{'vendor': 'New Company', 'amount': 25000}

FastAPI and Pydantic:
1. Read the request body.
2. Parse the JSON.
3. Validate the supplied fields.
4. Construct an InvoiceCreate object.
5. Pass that object into create_invoice().
6. Serialize the result into an HTTP response.

In [14]:
from fastapi import FastAPI, Header, Query
from pydantic import BaseModel

app = FastAPI()

class InvoiceUpdate(BaseModel):
    status: str

@app.patch("/invoices/{invoice_id}")
def update_invoice(
    invoice_id: int,
    payload: InvoiceUpdate,
    notify: bool = Query(default=False),
    x_request_id: str | None = Header(default=None)
):
    return {
        "invoice_id": invoice_id,
        "status": payload.status,
        "notify": notify,
        "request_id": x_request_id
    }

Build a FastAPI application exposing three endpoints:

1. GET  /health
2. GET  /agents/{agent_id}
3. POST /agents

In [ ]:

from fastapi import FastAPI, HTTPException, status
from pydantic import BaseModel, Field, field_validator

app = FastAPI(
    title="AI Agent Management API",
    version="1.0.0"
)

# Every incoming POST request must satisfy this scheme
# 1. Request schema
# for ex:
# {
#   "name": "Invoice Agent",
#   "model": "qwen2.5:7b",
#   "temperature": 0.2
# }
class AgentCreate(BaseModel):
    name: str = Field(min_length=1, max_length=100)
    model: str = Field(min_length=1)
    temperature: float = Field(ge=0, le=2)

    # @field_validator("name", "model") #rejects the white spaces
    # @classmethod
    # def validate_text(cls, value: str) -> str:
    #     value = value.strip()

    #     if not value:
    #         raise ValueError("Value cannot be blank")

    #     return value

In [ ]:
# 2. Response schema
class AgentResponse(AgentCreate): # Model Inheritance
    id: int

# AgentCreate
#   ├── name
#   ├── model
#   └── temperature
#          ↓
# AgentResponse
#   ├── name
#   ├── model
#   ├── temperature
#   └── id


# 3. In-memory storage
agents: dict[int, AgentResponse] = {}
next_agent_id = 1


# 4. Health check
@app.get("/health")
def health():
    return {"status": "healthy"}


# 5. Create an agent
@app.post(
    "/agents",
    response_model=AgentResponse,
    status_code=status.HTTP_201_CREATED
)
def create_agent(agent: AgentCreate):
    global next_agent_id

    new_agent = AgentResponse(
        id=next_agent_id,
        **agent.model_dump()
    )

    agents[next_agent_id] = new_agent
    next_agent_id += 1

    return new_agent


# 6. Retrieve an agent
@app.get(
    "/agents/{agent_id}",
    response_model=AgentResponse
)
def get_agent(agent_id: int):
    if agent_id not in agents:
        raise HTTPException(
            status_code=404,
            detail="Agent not found"
        )

    return agents[agent_id]


1. Uvicorn receives the HTTP request
2. FastAPI matches POST /agents
3. Pydantic validates AgentCreate
4. Field validators clean and check text
5. FastAPI invokes create_agent()
6. AgentResponse is constructed with id=1
7. The agent is stored in the dictionary
8. FastAPI validates and serializes the response

In [18]:

from fastapi import FastAPI, HTTPException, status
from pydantic import BaseModel, Field, field_validator

app = FastAPI(title="AI Agent Management API")


# ------------------------------
# 1. REQUEST MODEL
# ------------------------------

class AgentCreate(BaseModel):
    name: str = Field(min_length=1, max_length=100)
    model: str = Field(min_length=1)
    temperature: float = Field(ge=0, le=2)

    @field_validator("name", "model")
    @classmethod
    def validate_text(cls, value: str) -> str:
        value = value.strip()

        if not value:
            raise ValueError("Field cannot be blank")

        return value


# ------------------------------
# 2. PARTIAL UPDATE MODEL
# ------------------------------

class AgentUpdate(BaseModel):
    name: str | None = Field(
        default=None,
        min_length=1,
        max_length=100
    )

    model: str | None = Field(
        default=None,
        min_length=1
    )

    temperature: float | None = Field(
        default=None,
        ge=0,
        le=2
    )

    @field_validator("name", "model")
    @classmethod
    def validate_text(cls, value: str | None):
        if value is not None:
            value = value.strip()

            if not value:
                raise ValueError("Field cannot be blank")

        return value


# ------------------------------
# 3. RESPONSE MODEL
# ------------------------------

class AgentResponse(AgentCreate):
    id: int


# ------------------------------
# 4. IN-MEMORY STORAGE
# ------------------------------

agents: dict[int, AgentResponse] = {}
next_agent_id = 1


# ------------------------------
# 5. HEALTH CHECK
# ------------------------------

@app.get("/health")
def health():
    return {"status": "healthy"}


# ------------------------------
# 6. CREATE AGENT
# ------------------------------

@app.post(
    "/agents",
    response_model=AgentResponse,
    status_code=201
)
def create_agent(agent: AgentCreate):
    global next_agent_id

    new_agent = AgentResponse(
        id=next_agent_id,
        **agent.model_dump()
    )

    agents[next_agent_id] = new_agent
    next_agent_id += 1

    return new_agent


# ------------------------------
# 7. LIST ALL AGENTS
# ------------------------------

@app.get(
    "/agents",
    response_model=list[AgentResponse]
)
def get_all_agents():
    return list(agents.values())


# ------------------------------
# 8. GET ONE AGENT
# ------------------------------

@app.get(
    "/agents/{agent_id}",
    response_model=AgentResponse
)
def get_agent(agent_id: int):
    if agent_id not in agents:
        raise HTTPException(
            status_code=404,
            detail="Agent not found"
        )

    return agents[agent_id]


# ------------------------------
# 9. PARTIAL UPDATE
# ------------------------------

@app.patch(
    "/agents/{agent_id}",
    response_model=AgentResponse
)
def update_agent(
    agent_id: int,
    update: AgentUpdate
):
    if agent_id not in agents:
        raise HTTPException(
            status_code=404,
            detail="Agent not found"
        )

    existing_agent = agents[agent_id]

    update_data = update.model_dump(
        exclude_unset=True
    )

    if any(value is None for value in update_data.values()):
        raise HTTPException(
            status_code=422,
            detail="Updated fields cannot be null"
        )

    updated_agent = existing_agent.model_copy(
        update=update_data
    )

    agents[agent_id] = updated_agent

    return updated_agent


# ------------------------------
# 10. DELETE AGENT
# ------------------------------

@app.delete(
    "/agents/{agent_id}",
    status_code=status.HTTP_204_NO_CONTENT
)
def delete_agent(agent_id: int):
    if agent_id not in agents:
        raise HTTPException(
            status_code=404,
            detail="Agent not found"
        )

    del agents[agent_id]
